# Project : Empathetic Response AI

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

A chatbot that ignores how someone is feeling comes across as cold, especially in support chats. Companies like Zendesk lean toward AI that reads the emotional tone first and replies accordingly. Here we use a pretrained emotion classifier to detect the emotion in a message, then a LoRA-fine-tuned GPT-2 generates a reply that's aware of that emotion.

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q transformers datasets peft accelerate evaluate openai

import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling, pipeline
)
from peft import LoraConfig, get_peft_model, TaskType

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
Enter your Groq API key: ··········


### SECTION 2: Load Empathetic Dialogues Dataset (with robust fallback)

In [2]:
# quick heads up: "empathetic_dialogues" uses HuggingFace's old loading-script
# way of doing things (it grabs a tar file from an outside URL), and newer
# versions of the `datasets` library don't allow that by default anymore.
# so we try it, then a parquet backup, then just fall back to some made-up
# conversations we wrote ourselves if both of those don't work.
def robust_load_dataset(repo_candidates, split="train"):
    for repo_id in repo_candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', split='{split}')...")
            ds = load_dataset(repo_id, split=split)
            print(f"[dataset] Loaded '{repo_id}' successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' standard load failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/default/{split}/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback for '{repo_id}' succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback for '{repo_id}' failed: {type(e).__name__}: {e}")
    return None

raw_ds = robust_load_dataset(["empathetic_dialogues", "facebook/empathetic_dialogues"], split="train")

if raw_ds is not None:
    df = pd.DataFrame(raw_ds)[["context", "prompt", "utterance"]].dropna()
else:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_DIALOGUES = [
        {"context": "sad", "prompt": "I just lost my job after five years at the company.", "utterance": "I'm so sorry, that must feel really unsettling. Do you want to talk about what happened?"},
        {"context": "afraid", "prompt": "I have a big surgery scheduled next week and I'm terrified.", "utterance": "That sounds really scary to sit with. It's completely understandable to feel afraid before something like that."},
        {"context": "proud", "prompt": "I finally finished my degree after going back to school at 35.", "utterance": "That's amazing! Going back and finishing shows real determination -- you should be proud."},
        {"context": "angry", "prompt": "My roommate keeps eating my food without asking.", "utterance": "That's frustrating, especially when it keeps happening. Have you been able to talk to them about it?"},
        {"context": "lonely", "prompt": "I moved to a new city and don't know anyone yet.", "utterance": "Moving somewhere new can feel really isolating at first. It does get easier as you find your people."},
        {"context": "excited", "prompt": "I just got accepted into my dream graduate program!", "utterance": "That's fantastic news, congratulations! All that hard work really paid off."},
        {"context": "grateful", "prompt": "My neighbor helped me shovel my driveway during the snowstorm.", "utterance": "That's really kind of them. It's nice when people show up for each other like that."},
        {"context": "anxious", "prompt": "I have a big presentation tomorrow and can't stop overthinking it.", "utterance": "Presentations can bring up a lot of nerves. What part of it is worrying you the most?"},
        {"context": "disappointed", "prompt": "I didn't get the promotion I was working toward all year.", "utterance": "That's a tough one, especially after putting in so much effort. It's okay to feel let down."},
        {"context": "joyful", "prompt": "My best friend is visiting me for the first time in two years.", "utterance": "That's wonderful! Reunions like that are so special after being apart for so long."},
    ]
    expanded = SYNTHETIC_DIALOGUES * 30  # replicate so the Trainer has a workable number of steps
    df = pd.DataFrame(expanded)
    print("[dataset] Using synthetic fallback: swap in a real Kaggle/HF dataset before reporting final metrics.")

df = df.sample(n=min(30000, len(df)), random_state=42, replace=len(df) < 30000).reset_index(drop=True)
print("Training examples:", len(df))
print(df.head(2))

[dataset] Trying load_dataset('empathetic_dialogues', split='train')...


README.md:   0%|          | 0.00/7.15k [00:00<?, ?B/s]

empathetic_dialogues.py:   0%|          | 0.00/4.51k [00:00<?, ?B/s]

[dataset] 'empathetic_dialogues' standard load failed: RuntimeError: Dataset scripts are no longer supported, but found empathetic_dialogues.py
[dataset] Trying Parquet fallback: hf://datasets/empathetic_dialogues@refs/convert/parquet/default/train/*.parquet
[dataset] Parquet fallback for 'empathetic_dialogues' failed: HfUriError: Invalid HF URI 'hf://datasets/empathetic_dialogues@refs/convert/parquet/default/train/*.parquet'. Repository id must be 'namespace/name', got 'empathetic_dialogues'.
[dataset] Trying load_dataset('facebook/empathetic_dialogues', split='train')...


README.md:   0%|          | 0.00/7.15k [00:00<?, ?B/s]

...

[dataset] 'facebook/empathetic_dialogues' standard load failed: RuntimeError: Dataset scripts are no longer supported, but found empathetic_dialogues.py
[dataset] Trying Parquet fallback: hf://datasets/facebook/empathetic_dialogues@refs/convert/parquet/default/train/*.parquet


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 5.86MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

[dataset] Parquet fallback for 'facebook/empathetic_dialogues' succeeded (76673 rows).
Training examples: 30000
   context                                             prompt  \
0  content  There's nothing like being in your house after...   
1   caring              i loved taking care of my sisters pet   

                                           utterance  
0  There's nothing like being in your house after...  
1                                                yep  


### SECTION 3: Build an Emotion Classifier for Routing

In [3]:
# We reuse a free, pretrained emotion classifier to TAG each incoming
# message with an emotion label before generation -- this is what
# makes the pipeline "emotion-aware" rather than a plain chatbot.
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

def detect_emotion(text):
    result = emotion_classifier(text[:512])[0][0]
    return result["label"], result["score"]

sample_emotion, sample_score = detect_emotion(df["prompt"].iloc[0])
print(f"Detected emotion: {sample_emotion} ({sample_score:.2f})")

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Detected emotion: joy (0.98)


### SECTION 4: Format Emotion-Conditioned Training Pairs

In [4]:
def format_example(row):
    emotion, _ = detect_emotion(row["prompt"])
    text = f"<emotion={emotion}> User: {row['prompt']}\nAI:"
    full = text + " " + row["utterance"]
    return {"text": full}

# NOTE: running detect_emotion on every row is slow -- for full 30K rows,
# batch this with the pipeline's batch_size param in production.
formatted = df.sample(5000, random_state=1).apply(format_example, axis=1, result_type="expand")
dataset = Dataset.from_pandas(formatted).train_test_split(test_size=0.1, seed=42)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


### SECTION 5: Tokenize

In [5]:
model_checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [7]:
!pip uninstall -y torchao
!pip install -U peft transformers accelerate
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.1 MB/s eta 0:00:00


### SECTION 6: Apply LoRA to GPT-2

In [8]:
base_model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["c_attn"]  # GPT-2's combined attention projection
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


### SECTION 7: Fine-Tune

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = TrainingArguments(
    output_dir="./empathetic-ai-checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,      # kept small so free Colab doesn't run out of RAM
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    gradient_accumulation_steps=4,   # makes up for the smaller batch size above
    gradient_checkpointing=True,     # uses less memory, trains a little slower
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
    data_collator=data_collator
)
trainer.train()

GPU available: True


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,2.380210
2,2.690848,2.333773
3,2.690848,2.316044
4,2.420490,2.310934


TrainOutput(global_step=1128, training_loss=2.540294457834663, metrics={'train_runtime': 451.4013, 'train_samples_per_second': 39.876, 'train_steps_per_second': 2.499, 'total_flos': 1179891007488000.0, 'train_loss': 2.540294457834663, 'epoch': 4.0})

### SECTION 8: Build the Emotion-Aware Response Pipeline

In [10]:
def empathetic_response(user_message, max_new_tokens=60):
    emotion, score = detect_emotion(user_message)
    prompt = f"<emotion={emotion}> User: {user_message}\nAI:"
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **{k: v.to(model.device) for k, v in inputs.items()},
            max_new_tokens=max_new_tokens, do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    reply = tokenizer.decode(output[0], skip_special_tokens=True)
    return reply.split("AI:")[-1].strip(), emotion, score

reply, emotion, score = empathetic_response("I just lost my job and I don't know what to do.")
print(f"Detected emotion: {emotion} ({score:.2f})")
print(f"AI response: {reply}")

Detected emotion: sadness (0.76)
AI response: I'm sure I will have to work hard for my life to get back to the top.  I hope I can come back with an amazing job in life.  It's so hard to make it through this.  I hope I can go on with my life and make it back to the


### SECTION 9: Save Model

In [11]:
model.save_pretrained("./empathetic-ai-lora")
tokenizer.save_pretrained("./empathetic-ai-lora")
print("Model saved to ./empathetic-ai-lora")

Model saved to ./empathetic-ai-lora
